In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime
import os
import json
import requests
from bs4 import BeautifulSoup
import shutil
import time

In [ ]:
print("Script started, current directory:", os.getcwd())

TMDB_API_KEY = "dcbbf062e91bfc7ee36c9aa743dbba09"
TMDB_BASE_URL = "https://api.themoviedb.org/3"
TMDB_IMAGE_BASE_URL = "https://image.tmdb.org/t/p/w500"
OUTPUT_FILE = "modern_movies_2024_2025.csv"

def get_tmdb_full_data(tmdb_id):
    try:
        params = {"api_key": TMDB_API_KEY}
        # Main details
        url = f"{TMDB_BASE_URL}/movie/{tmdb_id}"
        response = requests.get(url, params=params)
        if response.status_code != 200:
            print(f"Failed to fetch main details for {tmdb_id}, status: {response.status_code}")
            return None
        data = response.json()

        # Keywords
        keywords_url = f"{TMDB_BASE_URL}/movie/{tmdb_id}/keywords"
        keywords_response = requests.get(keywords_url, params=params)
        keywords = keywords_response.json().get("keywords", []) if keywords_response.status_code == 200 else []

        # Credits (cast, director)
        credits_url = f"{TMDB_BASE_URL}/movie/{tmdb_id}/credits"
        credits_response = requests.get(credits_url, params=params)
        cast = []
        director = None
        if credits_response.status_code == 200:
            credits_data = credits_response.json()
            for member in credits_data.get("cast", []):
                cast.append({
                    "name": member.get("name"),
                    "profile_url": TMDB_IMAGE_BASE_URL + member["profile_path"] if member.get("profile_path") else None
                })
            for crew_member in credits_data.get("crew", []):
                if crew_member.get("job") == "Director":
                    director = crew_member.get("name")
                    break

        # Images
        images_url = f"{TMDB_BASE_URL}/movie/{tmdb_id}/images"
        images_response = requests.get(images_url, params=params)
        images = images_response.json().get("posters", []) if images_response.status_code == 200 else []
        poster_urls = [TMDB_IMAGE_BASE_URL + img["file_path"] for img in images if img.get("file_path")]

        # Videos
        videos_url = f"{TMDB_BASE_URL}/movie/{tmdb_id}/videos"
        videos_response = requests.get(videos_url, params=params)
        videos = videos_response.json().get("results", []) if videos_response.status_code == 200 else []
        video_keys = [video["key"] for video in videos if video.get("key")]

        # Reviews
        reviews_url = f"{TMDB_BASE_URL}/movie/{tmdb_id}/reviews"
        reviews_response = requests.get(reviews_url, params=params)
        reviews = reviews_response.json().get("results", []) if reviews_response.status_code == 200 else []
        review_texts = [review["content"] for review in reviews if review.get("content")]

        # External IDs
        external_url = f"{TMDB_BASE_URL}/movie/{tmdb_id}/external_ids"
        external_response = requests.get(external_url, params=params)
        external_ids = external_response.json() if external_response.status_code == 200 else {}

        # Release dates (for certification)
        release_url = f"{TMDB_BASE_URL}/movie/{tmdb_id}/release_dates"
        release_response = requests.get(release_url, params=params)
        certification = None
        if release_response.status_code == 200:
            results = release_response.json().get("results", [])
            for entry in results:
                if entry.get("iso_3166_1") == "US":
                    for rel in entry.get("release_dates", []):
                        if rel.get("certification"):
                            certification = rel["certification"]
                            break
                    if certification:
                        break

        result = {
            "movie_id": str(tmdb_id),
            "title": data.get("title"),
            "overview": data.get("overview"),
            "genres": [genre["name"] for genre in data.get("genres", [])],
            "keywords": [kw["name"] for kw in keywords],
            "director": director,
            "release_date": data.get("release_date"),
            "runtime": data.get("runtime"),
            "popularity": data.get("popularity"),
            "poster_url": TMDB_IMAGE_BASE_URL + data["poster_path"] if data.get("poster_path") else None,
            "budget": data.get("budget"),
            "revenue": data.get("revenue"),
            "tagline": data.get("tagline"),
            "poster_urls": poster_urls,
            "video_keys": video_keys,
            "review_texts": review_texts,
            "external_ids": external_ids,
            "certification": certification,
            "cast": cast,
        }
        return result
    except Exception as e:
        print(f"Error fetching data for {tmdb_id}: {e}")
        return None

def get_popular_movie_ids(year, max_pages=250):
    ids = []
    for page in range(1, max_pages + 1):
        url = f"{TMDB_BASE_URL}/discover/movie"
        params = {
            "api_key": TMDB_API_KEY,
            "sort_by": "popularity.desc",
            "primary_release_year": year,
            "page": page
        }
        try:
            response = requests.get(url, params=params)
            if response.status_code != 200:
                print(f"Failed to fetch discover page {page} for {year}, status: {response.status_code}")
                break
            data = response.json()
            for movie in data.get("results", []):
                if movie.get("id"):
                    ids.append(str(movie["id"]))
            if page >= data.get("total_pages", 1):
                break
            if len(ids) >= 10000:
                break
            if page % 10 == 0:
                print(f"Fetched {len(ids)} IDs for {year} (page {page})")
            time.sleep(0.2)
        except Exception as e:
            print(f"Error fetching IDs for {year} page {page}: {e}")
            break
    return ids[:10000]

all_ids = []
for year in [2024, 2025]:
    ids = get_popular_movie_ids(year, max_pages=250)
    print(f"Year {year}: {len(ids)} IDs")
    all_ids.extend(ids)

print(f"Total movies to fetch: {len(all_ids)}")
if not all_ids:
    print("No movie IDs found! Check your API key and network.")
    raise SystemExit

results = []
if os.path.exists(OUTPUT_FILE):
    df_existing = pd.read_csv(OUTPUT_FILE)
    done_ids = set(df_existing["movie_id"].astype(str))
    print(f"Resuming from backup, {len(done_ids)} movies already processed.")
else:
    df_existing = pd.DataFrame()
    done_ids = set()

start_idx = len(done_ids)
for idx, tmdb_id in enumerate(all_ids[start_idx:], start=start_idx):
    if tmdb_id in done_ids:
        continue
    movie_data = get_tmdb_full_data(tmdb_id)
    if movie_data:
        results.append(movie_data)
    if (idx + 1) % 10 == 0 or (idx + 1) == len(all_ids):
        temp_df = pd.DataFrame(results)
        df_existing = pd.concat([df_existing, temp_df], ignore_index=True)
        df_existing.to_csv(OUTPUT_FILE, index=False)
        results = []
        print(f"Saved backup at {len(df_existing)} movies.")
    if (idx + 1) % 50 == 0:
        print(f"Fetched {idx + 1} movies...")
    time.sleep(0.25)

print(f"Done! All modern movies processed and saved to {OUTPUT_FILE}.")